# MDPs and Bellman equations — interactive companion

Companion to [Post 1c: MDPs and Bellman](../posts/01c-mdps-and-bellman.qmd).
Bandits had one decision per episode. MDPs have *sequences* — each action
moves the agent into a new state, and rewards accumulate over time.

**What you'll do (≈ 20 minutes):**
1. Build a small gridworld and visualize its optimal value/policy.
2. Run value iteration and watch it converge.
3. Run policy iteration and compare.
4. Discover the honest result: PI needs fewer iterations, but VI is *faster wall-clock*.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 110

from nano_agents.mdp import (
    GridWorld, TwoGoalGridWorld,
    value_iteration, policy_iteration,
    bellman_optimality_residual,
)
from nano_agents.mdp.visualization import plot_policy, plot_value

## 1. The TwoGoalGridWorld

5×5 grid with two terminals: a small +1 at the top-right and a big +10
at the bottom-right. Walls in the middle force interesting routing.
Each step costs -0.04 (so the agent is rewarded for taking short paths).

In [ ]:
env = TwoGoalGridWorld(slip=0.15, step_reward=-0.04)
print(f"States: {env.nS}, actions: {env.nA}")
print(f"Slip probability: {env.slip}  (15% chance each step goes sideways)")
print(f"Terminals: {env.terminals}")

# Solve and visualize.
V, pi, _ = value_iteration(env, gamma=0.95)
fig, ax = plt.subplots(figsize=(8, 8))
plot_policy(env, pi, V=V, ax=ax)
ax.set_title("Optimal policy under stochastic transitions");
plt.show()

Look at the policy: even though +10 is at the bottom-right, the policy
near the +1 cell *also* points toward +1 — once you're that close, the
discount and step-cost don't justify trekking down to the bigger reward.

### Try this
- Set `slip=0.0` (deterministic). Does the policy change? Where does +1
  become preferable?
- Increase `step_reward` to `-0.5` (impatient agent). Does the boundary
  between "go to +1" and "go to +10" shift? Why?
- Change the +1 terminal to +9 (close to +10). What's the qualitative change?

## 2. Value iteration: watch it converge

Each VI step does one Bellman backup at every state simultaneously.
The Bellman optimality residual $\|TV - V\|_\infty$ should shrink by a
factor of $\gamma$ per iteration (the contraction property).

In [ ]:
# Manual VI with logging.
gamma = 0.95
V = np.zeros(env.nS)
P, R = env.transition_tensors()  # P[s, a, s'], R[s, a]
residuals = []
n_iters = 100
for it in range(n_iters):
    Q = R + gamma * (P @ V)
    V_new = Q.max(axis=1)
    # Force terminal states to keep their fixed value.
    for s in env.terminals:
        si = env.state_to_idx[s]
        V_new[si] = env.terminals[s]
    res = np.max(np.abs(V_new - V))
    residuals.append(res)
    V = V_new
    if res < 1e-9: break

plt.semilogy(residuals)
plt.xlabel("iteration")
plt.ylabel(r"$\|TV - V\|_\infty$  (log scale)")
plt.title(f"VI residual decay — slope ≈ log(γ) = {np.log(gamma):.3f}")
plt.grid(which="both", alpha=0.3); plt.show()
print(f"Converged in {len(residuals)} iterations.")

The log-residual is linear in iteration count — that's the contraction
property in a picture. Each iteration multiplies the error by $\gamma$
in the worst case.

### Try this
- Set `gamma = 0.5`. The contraction is faster — residuals plummet.
  But the policy considers fewer future steps. Compare the resulting policies.
- Set `gamma = 0.99`. Very slow contraction but very far-sighted policy.
  How many iterations to converge?

## 3. VI vs PI: an honest race

Policy iteration alternates two steps:
1. **Evaluate**: solve $V^\pi$ for the current policy (a linear system).
2. **Improve**: greedily set $\pi(s) = \arg\max_a Q^\pi(s, a)$.

It needs *far fewer outer iterations* than VI. But each PI iteration is
much more expensive (the policy evaluation step solves a linear system).
Which one wins on wall clock?

In [ ]:
# Use a larger problem to make the timing meaningful.
big = GridWorld(rows=10, cols=10,
                terminals={(0, 9): 1.0, (9, 9): 10.0, (5, 5): -5.0},
                walls={(2, 2), (2, 3), (2, 4), (5, 7), (6, 7), (7, 7)},
                step_reward=-0.04, slip=0.1)
print(f"Big problem: {big.nS} states.")

# Time VI.
t0 = time.perf_counter()
V_vi, pi_vi, n_vi = value_iteration(big, gamma=0.95, tol=1e-8)
t_vi = time.perf_counter() - t0
print(f"VI: {n_vi} iterations, wall time {t_vi*1000:.1f} ms")

# Time PI.
t0 = time.perf_counter()
V_pi, pi_pi, n_pi = policy_iteration(big, gamma=0.95, tol=1e-10)
t_pi = time.perf_counter() - t0
print(f"PI: {n_pi} outer iterations, wall time {t_pi*1000:.1f} ms")

# Do the policies agree?
agree = np.sum(pi_vi == pi_pi)
print(f"\\nPolicies agree on {agree}/{big.nS} states.")

**Honest result** (also in Post 1c §3): PI needs *much* fewer outer iterations
(maybe ~10× fewer), but the per-iteration cost of solving the linear system
$V^\pi = R^\pi + \gamma P^\pi V^\pi$ dominates. On most reasonably-sized
gridworlds, VI is the wall-clock winner.

PI dominates when:
- States are few (linear solve is cheap).
- You need *very high* accuracy (PI is exact at each evaluation).
- You're using an iterative inner solve that benefits from warm-starting.

VI dominates when:
- States are many (linear solve is expensive).
- You can stop early (e.g., once policy is stable).
- You have GPU-friendly batched operations.

### Try this
- Make the grid 20×20 (`rows=20, cols=20`). VI's win should grow.
- Use a 5×5 grid. PI might win here.
- Set `tol=1e-12` on VI. It needs many more iterations; PI doesn't change much.

## What's next

You can now solve any finite MDP if you have a model $(P, R)$. The next
problem is *what to do when you don't have a model* — you only see
samples from the environment.

That's Q-learning, in [`01d-pomdps-and-q-learning.ipynb`](01d-pomdps-and-q-learning.ipynb).
We'll also handle the case where you can't even fully observe the state.